In [ ]:
from pathlib import Path
import datetime

import torch
import torch.nn.functional as F
from jaxtyping import Int

# custom utils
from muutils.misc import shorten_numerical_to_str
from trnbl import TrainingManager

# from trnbl.loggers.local import LocalLogger
from trnbl.loggers.tensorboard import TensorBoardLogger


from attention_motifs.ae import AttnAEConfig, AttnAE, contrastive_loss
from attention_motifs.dataset.dataset import CollectedAttentionPatternDataloader
from attention_motifs.dataset.util import AttentionPatternMetadata

In [ ]:
# magic autoreload
%load_ext autoreload
%autoreload 2

In [ ]:
BATCH_SIZE: int = 4
N_TRAIN_BATCHES: int = 10
N_VAL_BATCHES: int = 10

In [ ]:
TRAIN_LOADER_DATASET = CollectedAttentionPatternDataloader.read(
	"../data/activations/pile_5"
)
VAL_LOADER_DATASET = CollectedAttentionPatternDataloader.read(
	"../data/activations/pile_5_val"
)

TRAIN_LOADER = TRAIN_LOADER_DATASET.dataloader(BATCH_SIZE, shuffle=True, max_batches=N_TRAIN_BATCHES)
VAL_LOADER = VAL_LOADER_DATASET.dataloader(BATCH_SIZE, shuffle=True, max_batches=N_VAL_BATCHES)

print(f"Train loader: {len(TRAIN_LOADER)} batches, {len(TRAIN_LOADER.dataset)} samples")

In [ ]:
MODEL: AttnAE = AttnAE(
	AttnAEConfig(
		latent_dim=128,
	)
)

MODEL_CONFIG: AttnAEConfig = MODEL.config

model_n_params: int = sum(p.numel() for p in MODEL.parameters())
print(
	f"model has {model_n_params} ({shorten_numerical_to_str(model_n_params)}) parameters"
)

# model

In [ ]:
TRAIN_LOADER: torch.utils.data.DataLoader
VAL_LOADER: torch.utils.data.DataLoader | None = None

PROJECT_NAME: str = "contrastive-ae"
CHECKPT_INTERVAL: str = "1/2 run"
EVAL_INTERVAL: str = "1/2 run"
DEVICE: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL = MODEL.to(DEVICE)

In [ ]:
OPTIMIZER: torch.optim.Optimizer
LR_SCHEDULER: torch.optim.lr_scheduler._LRScheduler
OPTIMIZER, LR_SCHEDULER = MODEL_CONFIG.get_optim_and_lrs(MODEL)

In [ ]:
def evaluation_step(model: AttnAE) -> dict[str, float]:
	"""Evaluate model on validation set"""
	if VAL_LOADER is None:
		return {}

	model.eval()
	val_metrics: dict[str, float] = {"val/loss": 0.0, "val/recon_loss": 0.0, "val/contrast_loss": 0.0}

	with torch.no_grad():
		for patterns, metadata in VAL_LOADER:
			patterns = patterns.to(DEVICE).to(torch.float32).unsqueeze(1)
			OPTIMIZER.zero_grad()
			x_recon, embeddings = model(patterns)

			# reconstruction loss
			recon_loss = F.mse_loss(x_recon, patterns)

			# contrastive loss using all pairs in batch
			# compute "classes" for contrastive loss
			# classes is a tensor of the same shape as the batch, where each element is an integer
			classes: Int[torch.Tensor, " batch"] = (
				AttentionPatternMetadata.contrastive_classes(metadata).to(DEVICE)
			)

			# compute contrastive loss
			contrast_loss = contrastive_loss(embeddings, classes, temperature=config.contrast_temperature)

			# combined loss and backward pass
			total_loss = (
				model.config.recon_weight * recon_loss 
				+ model.config.contrast_weight * contrast_loss
			)
			val_metrics["val/loss"] += total_loss.item()
			val_metrics["val/recon_loss"] += recon_loss.item()
			val_metrics["val/contrast_loss"] += contrast_loss.item()

	for k in val_metrics:
		val_metrics[k] /= len(VAL_LOADER)

	model.train()
	return val_metrics

In [ ]:
# setup logger
LOGGER: TensorBoardLogger = TensorBoardLogger(
	log_dir=Path("tb-logs-convAE"),
	name=PROJECT_NAME + datetime.datetime.now().strftime("-%Y-%m-%d-%H-%M-%S"),
	# metric_names=[
	# 	"train/loss",
	# 	"train/recon_loss",
	# 	"train/contrast_loss",
	# 	"val/loss",
	# 	"val/recon_loss",
	# 	"val/contrast_loss",
	# ],
	train_config=dict(
		model_config=MODEL.zanj_model_config.serialize(),
		model_str=str(MODEL),
	),
)

with TrainingManager(
	model=MODEL,
	logger=LOGGER,
	evals={
		EVAL_INTERVAL: evaluation_step,
	}.items(),
	checkpoint_interval=CHECKPT_INTERVAL,
) as tr:
	for epoch in tr.epoch_loop(range(MODEL_CONFIG.num_epochs)):
		for patterns, metadata in tr.batch_loop(TRAIN_LOADER):
			# move to device
			patterns = patterns.to(DEVICE).to(torch.float32).unsqueeze(1)

			# reset gradients
			OPTIMIZER.zero_grad()

			# forward pass
			x_recon, embeddings = MODEL(patterns)

			# reconstruction loss
			recon_loss = F.mse_loss(x_recon, patterns)

			# compute "classes" for contrastive loss
			# classes is a tensor of the same shape as the batch, where each element is an integer
			classes: Int[torch.Tensor, " batch"] = (
				AttentionPatternMetadata.contrastive_classes(metadata).to(DEVICE)
			)

			# compute contrastive loss
			contrast_loss = contrastive_loss(embeddings, classes)

			# combined loss and backward pass
			total_loss = (
				MODEL_CONFIG.recon_weight * recon_loss 
				+ MODEL_CONFIG.contrast_weight * contrast_loss
			)

			# backward pass
			total_loss.backward()
			OPTIMIZER.step()
			LR_SCHEDULER.step()

			# log metrics
			tr.batch_update(
				samples=len(metadata),
				**{
					"train/loss": total_loss.item(),
					"train/recon_loss": recon_loss.item(),
					"train/contrast_loss": contrast_loss.item(),
					"lr": LR_SCHEDULER.get_last_lr()[0],
				},
			)

			del patterns, x_recon, embeddings, recon_loss, contrast_loss, total_loss